# Vers?o 4 ? previs?o did?tica de cotas do Rio Taquari

Esta V4 parte do fluxo de importa??o e auditoria da V3, mantendo as vers?es anteriores intactas. O alvo ? a cota m?dia di?ria de amanh? (t+1), observada na esta??o fluviom?trica ANA 86720000 em Encantado. Trata-se de n?vel de rio, n?o de volume armazenado em barragem. A avalia??o ? retrospectiva e explorat?ria: o per?odo de teste j? apareceu em an?lises anteriores.

A regra de divis?o ? cronol?gica: 80% iniciais para desenvolvimento, sendo 85% desse bloco usados para ajuste e 15% para valida??o; 20% finais ficam reservados para teste. Isso corresponde aproximadamente a 68% treino efetivo + 12% valida??o + 20% teste.


## Ambiente, seed e diret?rios

Esta c?lula importa pandas, NumPy, Matplotlib, scikit-learn e TensorFlow, fixa seed 42 e cria `resultados/v4/`. A seed ajuda a repetir os sorteios, sem garantir igualdade entre ambientes. Esperamos ver vers?es instaladas e o caminho raiz; uma biblioteca ausente interrompe a execu??o.


In [ ]:
from pathlib import Path
import sys, warnings, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEMENTE = 42
np.random.seed(SEMENTE)
tf.keras.utils.set_random_seed(SEMENTE)
try:
    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass

def localizar_projeto():
    candidatos = [Path.cwd(), Path.cwd().parent]
    try: candidatos.append(Path(__file__).resolve().parents[1])
    except NameError: pass
    for pasta in candidatos:
        if (pasta / 'dados').is_dir() and (pasta / 'notebooks').is_dir(): return pasta.resolve()
    raise FileNotFoundError('Abra o projeto ou execute a partir da pasta RedesNeurais_ANA.')

RAIZ = localizar_projeto()
DIR_DADOS = RAIZ / 'dados'
DIR_RESULTADOS = RAIZ / 'resultados' / 'v4'
DIR_FIGURAS = DIR_RESULTADOS / 'figuras'
DIR_RESULTADOS.mkdir(parents=True, exist_ok=True)
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
VERSOES = {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'scikit-learn': sklearn.__version__, 'tensorflow': tf.__version__, 'seed': SEMENTE}
(DIR_RESULTADOS / 'versoes.json').write_text(json.dumps(VERSOES, indent=2), encoding='utf-8')
print(VERSOES)
print('Projeto:', RAIZ)
print('Saídas V2:', DIR_RESULTADOS)

## 2. Leitura e auditoria das exportações ANA

As linhas de metadados anteriores ao cabeçalho são mantidas para auditoria. A expansão diária respeita status `1`, `MediaDiaria=1` para cota e vazão, calendário real e prioridade de consistência `2`. Ausências nunca são convertidas em zero.

## Ler e auditar arquivos ANA

Esta c?lula l? os tr?s CSVs de cotas, chuva e vaz?o, identifica a codifica??o e separa metadados do cabe?alho. O relat?rio mostra cobertura e colunas encontradas. Confira os nomes e datas antes de avan?ar.


In [ ]:
ARQUIVOS = {'Cota': DIR_DADOS / '86720000_Cotas.csv', 'Vazao': DIR_DADOS / '86720000_Vazoes.csv', 'Chuva': DIR_DADOS / '2951010_Chuvas.csv'}
def ler_exportacao_ana(caminho):
    if not caminho.is_file(): raise FileNotFoundError(f'Arquivo obrigatório ausente: {caminho}')
    conteudo = caminho.read_bytes(); texto = None; codificacao = None
    for candidata in ('utf-8-sig', 'cp1252', 'latin1'):
        try: texto = conteudo.decode(candidata); codificacao = candidata; break
        except UnicodeDecodeError: continue
    linhas = texto.splitlines()
    indice = next((i for i, linha in enumerate(linhas) if linha.lstrip('\ufeff').startswith('EstacaoCodigo;')), None)
    if indice is None: raise ValueError(f'Cabeçalho ANA não encontrado em {caminho.name}')
    tabela = pd.read_csv(caminho, sep=';', encoding=codificacao, skiprows=indice, dtype='string', skipinitialspace=True)
    tabela.columns = tabela.columns.str.strip()
    return tabela, codificacao, indice + 1, linhas[:indice]

def para_numero(serie):
    return pd.to_numeric(serie.astype('string').str.strip().str.replace(',', '.', regex=False), errors='coerce')

brutos, auditoria_arquivos, metadados = {}, [], {}
for nome, caminho in ARQUIVOS.items():
    tabela, codificacao, linha_cabecalho, linhas_meta = ler_exportacao_ana(caminho)
    brutos[nome] = tabela; metadados[nome] = linhas_meta
    datas = pd.to_datetime(tabela['Data'], dayfirst=True, errors='coerce')
    col_dia = {'Cota':'Cota01', 'Vazao':'Vazao01', 'Chuva':'Chuva01'}[nome]
    auditoria_arquivos.append({'Série':nome,'Arquivo':caminho.name,'Codificação':codificacao,'Linha cabeçalho':linha_cabecalho,'Linhas mensais':len(tabela),'Colunas':len(tabela.columns),'Primeira data':datas.min(),'Última data':datas.max(),'Coluna diária presente':col_dia in tabela})
display(pd.DataFrame(auditoria_arquivos))
print('Metadados de cotas:', *metadados['Cota'], sep='\n')

## 3. Expandir os dados para dias

**O que faz:** Expande colunas diarias; mantem status real e MediaDiaria=1 para cota/vazao. Prioriza nivel de consistencia e usa mediana em empates. Zero de chuva e valido; vazio nao e zero. Confira lacunas e duplicatas.

**Por que:** preparar ou interpretar a previsao diaria e os episodios do rio.

**Entradas/saidas e verificacao:** usa objetos/csvs mencionados na etapa; confira datas, contagens, lacunas, escalas e avisos antes de interpretar.

In [ ]:
def expandir_diario(nome, tabela):
    prefixo = nome; mensal = tabela.copy()
    mensal['Data'] = pd.to_datetime(mensal['Data'], dayfirst=True, errors='coerce')
    mensal['NivelConsistencia'] = para_numero(mensal['NivelConsistencia'])
    if nome in ('Cota','Vazao'):
        mensal['MediaDiaria'] = para_numero(mensal['MediaDiaria'])
        mensal = mensal[mensal['MediaDiaria'].eq(1)]
    blocos = []
    for dia in range(1,32):
        coluna = f'{prefixo}{dia:02d}'; status_col = f'{coluna}Status'
        if coluna not in mensal: continue
        data_dia = mensal['Data'] + pd.to_timedelta(dia-1, unit='D')
        bloco = pd.DataFrame({'Data':data_dia,'Valor':para_numero(mensal[coluna]),'Status':mensal[status_col].astype('string').str.strip() if status_col in mensal else pd.NA,'Consistencia':mensal['NivelConsistencia']})
        blocos.append(bloco.loc[data_dia.dt.month.eq(mensal['Data'].dt.month)])
    longo = pd.concat(blocos, ignore_index=True).dropna(subset=['Data','Valor'])
    elegivel = longo[longo['Status'].eq('1') & np.isfinite(longo['Valor'])].copy()
    if elegivel.empty: raise ValueError(f'Sem valores elegíveis para {nome}')
    colisoes = int(elegivel.groupby('Data').size().gt(1).sum())
    elegivel = elegivel.sort_values(['Data','Consistencia'], ascending=[True,False])
    melhor = elegivel.groupby('Data')['Consistencia'].transform('max')
    elegivel = elegivel[elegivel['Consistencia'].eq(melhor)]
    diaria = elegivel.groupby('Data')['Valor'].median().sort_index().rename(nome)
    resumo = {'Série':nome,'Observações com valor':len(longo),'Elegíveis após status':len(elegivel),'Dias únicos':len(diaria),'Dias com duplicatas elegíveis':colisoes,'Início':diaria.index.min(),'Fim':diaria.index.max(),'Lacunas':int((diaria.index.to_series().diff().dt.days > 1).sum())}
    return diaria, resumo, longo

series, resumos, dados_longos = {}, [], {}
for nome, tabela in brutos.items():
    series[nome], resumo, dados_longos[nome] = expandir_diario(nome,tabela); resumos.append(resumo)
display(pd.DataFrame(resumos))

base = pd.concat(series.values(), axis=1).sort_index(); base.index.name = 'Data'
base.to_csv(DIR_RESULTADOS / 'series_diarias.csv', date_format='%Y-%m-%d')
calendario = pd.date_range(base.index.min(), base.index.max(), freq='D', name='Data')
diaria = base.reindex(calendario)
print('Cobertura série diária:'); display(base.notna().sum().rename('dias observados').to_frame())

# Auditoria específica do evento de maio de 2024: valor e status nas linhas originais.
def auditoria_evento(tabela, coluna, data='01/05/2024'):
    campos = [c for c in ['Data','NivelConsistencia','hora','Hora','MediaDiaria','TipoMedicaoCotas',coluna,f'{coluna}Status'] if c in tabela.columns]
    return tabela.loc[tabela['Data'].eq(data), campos]
evento_cota = auditoria_evento(brutos['Cota'],'Cota02')
evento_chuva = auditoria_evento(brutos['Chuva'],'Chuva02')
display(evento_cota); display(evento_chuva)
evento_cota.to_csv(DIR_RESULTADOS / 'auditoria_evento_maio_2024_cota.csv', index=False)
evento_chuva.to_csv(DIR_RESULTADOS / 'auditoria_evento_maio_2024_chuva.csv', index=False)

## 3. Constru??o de janelas v?lidas e divis?o temporal

Para cada janela, as cotas usam `t?(n?1)?t`. No cen?rio de chuva at? `t`, a chuva usa os mesmos dias; no cen?rio conservador, usa `t?n?t?1`. O alvo ? sempre a cota de `t+1`. J7/J15/J30 compartilham a interse??o de datas eleg?veis. Para preservar a avalia??o explorat?ria da V1, usamos seus limites exatos: treino at? 09/03/1993, valida??o de 10/03/1993 a 24/11/2008 e teste a partir de 25/11/2008.

## Construir janelas e cortes cronol?gicos

Esta c?lula monta exemplos com janelas consecutivas de 7, 15 e 30 dias; nenhuma lacuna ? preenchida. O alvo ? a cota do dia seguinte. As datas s?o alinhadas entre janelas e os 20% finais ficam no teste; dos 80% iniciais, 15% s?o valida??o. Os contadores permitem conferir o suporte de cada parte.


In [ ]:
JANELAS = (7,15,30)
cota_array = diaria['Cota'].to_numpy(dtype=float, na_value=np.nan)
chuva_array = diaria['Chuva'].to_numpy(dtype=float, na_value=np.nan)
finite_cota = np.isfinite(cota_array)
finite_chuva = np.isfinite(chuva_array)

def mascara_janela(valida, n):
    cumulativa = np.r_[0, np.cumsum(valida, dtype=np.int64)]
    fim = np.arange(len(valida))
    inicio = fim - n + 1
    ok = inicio >= 0
    ok[ok] &= (cumulativa[fim[ok]+1] - cumulativa[inicio[ok]]) == n
    return ok

def construir_amostras(n, fim_chuva):
    deslocamento = 0 if fim_chuva == 't' else 1
    # i ? t; para o protocolo t-1, a chuva termina em i-1.
    chuva_ok = mascara_janela(finite_chuva, n)
    if deslocamento:
        chuva_ok = np.r_[False, chuva_ok[:-1]]
    cota_ok = mascara_janela(finite_cota, n)
    alvo_ok = np.r_[finite_cota[1:], False]
    valido = cota_ok & chuva_ok & alvo_ok
    valido[:n-1+deslocamento] = False
    indices = np.flatnonzero(valido)
    linhas = []
    for i in indices:
        inicio_cota = i-n+1
        fim_chuva_idx = i-deslocamento
        inicio_chuva = fim_chuva_idx-n+1
        linhas.append({'t':calendario[i], 'data_alvo':calendario[i+1],
                       'cotas':cota_array[inicio_cota:i+1].copy(),
                       'chuvas':chuva_array[inicio_chuva:fim_chuva_idx+1].copy(),
                       'alvo':float(cota_array[i+1]), 'persistencia':float(cota_array[i])})
    return linhas

amostras = {(n,c):construir_amostras(n,c) for n in JANELAS for c in ('t','t-1')}
contagens_elegiveis = {(n,c):len(amostras[(n,c)]) for n in JANELAS for c in ('t','t-1')}
for c in ('t','t-1'):
    sets = [set(pd.Timestamp(r['t']) for r in amostras[(n,c)]) for n in JANELAS]
    comuns = set.intersection(*sets)
    for n in JANELAS: amostras[(n,c)] = [r for r in amostras[(n,c)] if pd.Timestamp(r['t']) in comuns]
    print(f'Chuva at? {c}: {len(comuns):,} datas comuns; eleg?veis antes da interse??o: {[contagens_elegiveis[(n,c)] for n in JANELAS]}')

def empacotar(linhas, n, uso_chuva):
    features = [f'cota_d{i-n+1}' for i in range(n)]
    if uso_chuva: features += [f'chuva_d{i-n+1}' for i in range(n)]
    X = np.vstack([np.r_[r['cotas'], r['chuvas'] if uso_chuva else []] for r in linhas])
    return X, np.array([r['alvo'] for r in linhas]), np.array([r['persistencia'] for r in linhas]), pd.DatetimeIndex([r['t'] for r in linhas]), pd.DatetimeIndex([r['data_alvo'] for r in linhas]), features

def dividir(X,y,persist,t,alvos,features):
    # Os 20% finais s?o teste. A valida??o ocupa 15% dos 80% iniciais.
    ordem=np.argsort(alvos.values if hasattr(alvos,'values') else alvos)
    n=len(ordem); corte_teste=int(np.floor(0.80*n)); corte_treino=int(np.floor(0.85*corte_teste))
    masks={'treino':np.zeros(n,dtype=bool),'validacao':np.zeros(n,dtype=bool),'teste':np.zeros(n,dtype=bool)}
    masks['treino'][ordem[:corte_treino]]=True
    masks['validacao'][ordem[corte_treino:corte_teste]]=True
    masks['teste'][ordem[corte_teste:]]=True
    return {nome:{'X':X[m],'y':y[m],'persist':persist[m],'t':t[m],'alvos':alvos[m],'features':features} for nome,m in masks.items()}


particoes = {} ; cobertura = []
for chuva_fim in ('t','t-1'):
  for n in JANELAS:
    linhas=amostras[(n,chuva_fim)]
    for com_chuva in (False,True):
      X,y,p,t,alvos,features=empacotar(linhas,n,com_chuva)
      particoes[(chuva_fim,n,com_chuva)] = dividir(X,y,p,t,alvos,features)
    p=particoes[(chuva_fim,n,False)]
    cobertura.append({'Chuva termina em':chuva_fim,'Janela':n,'N comum':len(linhas),'Inicio t':p['treino']['t'][0],'Fim alvo':p['teste']['alvos'][-1],'N treino':len(p['treino']['y']),'N valida??o':len(p['validacao']['y']),'N teste':len(p['teste']['y'])})
cobertura = pd.DataFrame(cobertura); display(cobertura)
cobertura.to_csv(DIR_RESULTADOS / 'cobertura_e_splits.csv', index=False, date_format='%Y-%m-%d')

cobertura_anual=[]
for c in ('t','t-1'):
  for n in JANELAS:
    anos=pd.Series([r['data_alvo'].year for r in amostras[(n,c)]]).value_counts().sort_index()
    cobertura_anual.extend({'Chuva termina em':c,'Janela':n,'Ano':ano,'N':qtd} for ano,qtd in anos.items())
pd.DataFrame(cobertura_anual).to_csv(DIR_RESULTADOS / 'cobertura_anual.csv',index=False)


## 4. Seleção de modelos e avaliação exploratória

Em cada janela e cenário, são testadas MLPs 32 e 64–32 para cotas e para cotas+chuva. Early stopping monitora validação. Persistência e Ridge são avaliados nas mesmas datas. O modelo final é escolhido pela MAE de validação separadamente para cada combinação de janela e conjunto de entradas.

## Treinar MLP, Ridge e persist?ncia

Esta c?lula compara arquiteturas MLP, Ridge e persist?ncia. Os scalers s?o ajustados no treino; early stopping monitora a valida??o. Seed 42, Adam com taxa 0,001, Huber, lotes de 128, at? 120 ?pocas e paci?ncia 12 s?o par?metros herdados do fluxo V3. O teste s? ? usado ap?s selecionar a arquitetura. A sa?da cont?m m?tricas, previs?es e hist?rico de perdas.


In [ ]:
def criar_mlp(n_features, arquitetura):
    camadas=[keras.layers.Input(shape=(n_features,))]
    camadas += [keras.layers.Dense(u,activation='relu') for u in arquitetura]
    camadas += [keras.layers.Dense(1,activation='linear')]
    modelo=keras.Sequential(camadas)
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),loss=keras.losses.Huber())
    return modelo
def metricas(real,pred):
    return {'MAE':mean_absolute_error(real,pred),'RMSE':np.sqrt(mean_squared_error(real,pred)),'R2':r2_score(real,pred)}

resultados_modelos=[]; detalhes_modelos=[]; previsoes={}; historicos=[]
for chuva_fim in ('t','t-1'):
  for n in JANELAS:
    for com_chuva in (False,True):
      p=particoes[(chuva_fim,n,com_chuva)]; tr,va,te=p['treino'],p['validacao'],p['teste']
      sx=MinMaxScaler(); sy=MinMaxScaler()
      xtr=sx.fit_transform(tr['X']); xva=sx.transform(va['X']); xte=sx.transform(te['X'])
      ytr=sy.fit_transform(tr['y'].reshape(-1,1)); yva=sy.transform(va['y'].reshape(-1,1))
      family='cota+chuva' if com_chuva else 'cota'
      candidatos=[]
      for arquitetura in ((),(32,),(64,32)):
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEMENTE)
        modelo=criar_mlp(xtr.shape[1],arquitetura)
        parada=keras.callbacks.EarlyStopping(monitor='val_loss',patience=12,min_delta=1e-5,restore_best_weights=True)
        hist=modelo.fit(xtr,ytr,validation_data=(xva,yva),epochs=120,batch_size=128,shuffle=False,callbacks=[parada],verbose=0)
        predva=sy.inverse_transform(modelo.predict(xva,verbose=0)).ravel()
        maeva=mean_absolute_error(va['y'],predva)
        nome=f'MLP {arquitetura} {family}'
        candidatos.append({'modelo':modelo,'sx':sx,'sy':sy,'nome':nome,'arquitetura':arquitetura,'mae_val':maeva,'hist':hist.history,'epocas':len(hist.history['loss'])})
      escolhido=min(candidatos,key=lambda z:z['mae_val'])
      for cand in candidatos:
        detalhes_modelos.append({'Chuva termina em':chuva_fim,'Janela':n,'Entradas':family,'Arquitetura':str(cand['arquitetura']),'MAE validação':cand['mae_val'],'Épocas':cand['epocas'],'Selecionado':cand is escolhido,'N treino':len(tr['y']),'N validação':len(va['y']),'N teste':len(te['y']),'Início treino':str(tr['alvos'][0].date()),'Fim treino':str(tr['alvos'][-1].date()),'Início validação':str(va['alvos'][0].date()),'Fim validação':str(va['alvos'][-1].date()),'Início teste':str(te['alvos'][0].date()),'Fim teste':str(te['alvos'][-1].date())})
        for ep,(lt,lv) in enumerate(zip(cand['hist']['loss'],cand['hist']['val_loss']),1): historicos.append({'Chuva termina em':chuva_fim,'Janela':n,'Entradas':family,'Arquitetura':str(cand['arquitetura']),'Época':ep,'Perda treino':lt,'Perda validação':lv})
      pred=escolhido['sy'].inverse_transform(escolhido['modelo'].predict(xte,verbose=0)).ravel()
      chave=(chuva_fim,n,com_chuva); previsoes[chave]={'real':te['y'],'pred':pred,'datas':te['alvos'],'treino':tr,'teste':te,'modelo':escolhido}
      r=metricas(te['y'],pred); resultados_modelos.append({'Protocolo':f'Chuva até {chuva_fim}','Janela':n,'Modelo':escolhido['nome'],'Entradas':family,'N treino':len(tr['y']),'N validação':len(va['y']),'N teste':len(te['y']),**r})
      pred_p=te['persist']; rp=metricas(te['y'],pred_p); resultados_modelos.append({'Protocolo':f'Chuva até {chuva_fim}','Janela':n,'Modelo':'Persistência','Entradas':'cota(t)','N treino':len(tr['y']),'N validação':len(va['y']),'N teste':len(te['y']),**rp})
      ridge=Ridge(alpha=1.0); ridge.fit(sx.transform(tr['X']),tr['y']); pred_r=ridge.predict(xte); rr=metricas(te['y'],pred_r)
      resultados_modelos.append({'Protocolo':f'Chuva até {chuva_fim}','Janela':n,'Modelo':f'Ridge {family}','Entradas':family,'N treino':len(tr['y']),'N validação':len(va['y']),'N teste':len(te['y']),**rr})
      previsoes[chave]['ridge']=pred_r; previsoes[chave]['persist']=pred_p

df_resultados=pd.DataFrame(resultados_modelos); df_detalhes=pd.DataFrame(detalhes_modelos); df_hist=pd.DataFrame(historicos)
display(df_resultados); display(df_detalhes)
df_resultados.to_csv(DIR_RESULTADOS/'metricas_modelos.csv',index=False)
df_detalhes.to_csv(DIR_RESULTADOS/'selecao_modelos_validacao.csv',index=False)
df_hist.to_csv(DIR_RESULTADOS/'historico_treinamento.csv',index=False)
for c in ('t','t-1'):
  tabs=[]
  for n in JANELAS:
    v=previsoes[(c,n,True)]
    parte=pd.DataFrame({'janela':n,'data_alvo':v['datas'],'cota_real':v['real'],'mlp_cota_chuva':v['pred'],'ridge':v['ridge'],'persistencia':v['persist']})
    tabs.append(parte)
  pd.concat(tabs,ignore_index=True).to_csv(DIR_RESULTADOS/f'predicoes_teste_chuva_{c}.csv',index=False,date_format='%Y-%m-%d')

## 5. Figuras e análise por nível

Os gráficos principais usam o protocolo de chuva até `t`, com as mesmas datas de teste entre janelas. A unidade é mantida como aparece nas exportações; a conversão para metros só é usada no comentário de comparação com o levantamento do SGB e não é aplicada às métricas sem validação formal da unidade. P10/P90 são calculados somente no treino.

## Gr?ficos de previs?o

Esta etapa re?ne as previs?es j? calculadas para comparar as janelas. Observe alinhamento temporal, escala dos eixos e diferen?a entre observado e previsto. As figuras devem ser interpretadas junto das m?tricas, sem inferir desempenho operacional.


In [ ]:
# As três janelas no protocolo escolhido por convenção de disponibilidade mais ampla: chuva até t.
pred_todas={n:previsoes[('t',n,True)] for n in JANELAS}
datas_comuns=set.intersection(*[set(v['datas']) for v in pred_todas.values()])
datas_comuns=pd.DatetimeIndex(sorted(datas_comuns))
for n in JANELAS:
    v=pred_todas[n]; idx=pd.Index(v['datas']).get_indexer(datas_comuns);
    fig,ax=plt.subplots(figsize=(13,5)); ax.plot(datas_comuns,v['real'][idx],label='Cota observada',lw=1.1); ax.plot(datas_comuns,v['pred'][idx],label='MLP prevista',lw=1.0)
    ax.set(title=f'Cota observada e previsão — janela {n} dias (teste exploratório)',xlabel='Data do alvo',ylabel='Cota (unidade da exportação ANA)'); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(DIR_FIGURAS/f'comparacao_{n}_dias.png',dpi=160,bbox_inches='tight'); plt.show()

fig,axes=plt.subplots(1,3,figsize=(15,4))
metricas_fig=[('MAE','MAE (menor é melhor)'),('RMSE','RMSE (menor é melhor)'),('R2','R² (maior é melhor)')]
for ax,(col,titulo) in zip(axes,metricas_fig):
    q=df_resultados[(df_resultados['Protocolo']=='Chuva até t') & (df_resultados['Entradas']=='cota+chuva') & (df_resultados['Modelo'].str.startswith('MLP'))].sort_values('Janela')
    ax.plot(q['Janela'],q[col],marker='o'); ax.set(title=titulo,xlabel='Janela (dias)',xticks=JANELAS); ax.grid(alpha=.25)
fig.suptitle('Comparação das janelas — MLP com cotas e chuva'); fig.tight_layout(); fig.savefig(DIR_FIGURAS/'comparacao_janelas_metricas.png',dpi=160,bbox_inches='tight'); plt.show()

# Sobreposição das previsões selecionadas no mesmo episódio/datas.
fig,ax=plt.subplots(figsize=(13,5));
for n,v in pred_todas.items():
    idx=pd.Index(v['datas']).get_indexer(datas_comuns); ax.plot(datas_comuns,v['pred'][idx],label=f'J{n}',lw=.9)
v=pred_todas[7]; idx=pd.Index(v['datas']).get_indexer(datas_comuns); ax.plot(datas_comuns,v['real'][idx],color='black',label='Observado',lw=1.2)
ax.set(title='Sobreposição das previsões nas mesmas datas de teste',xlabel='Data do alvo',ylabel='Cota (unidade da exportação ANA)'); ax.grid(alpha=.25); ax.legend(ncol=4); fig.tight_layout(); fig.savefig(DIR_FIGURAS/'sobreposicao_janelas.png',dpi=160,bbox_inches='tight'); plt.show()

# Erros por faixa estatística e resíduos com sinal para J7 com chuva até t.
v=pred_todas[7]; p=particoes[('t',7,True)]['treino']; p10,p90=np.quantile(p['y'],[.10,.90]); erro=v['pred']-v['real']
faixa=np.select([v['real']<=p10,v['real']>=p90],['Baixa (≤ P10 treino)','Alta (≥ P90 treino)'],'Central')
residuos=pd.DataFrame({'Data':v['datas'],'Real':v['real'],'Previsto':v['pred'],'Erro previsto−real':erro,'Faixa':faixa})
resumo_erros=residuos.groupby('Faixa',observed=True).apply(lambda g:pd.Series({'N':len(g),'MAE':np.mean(np.abs(g['Erro previsto−real'])),'RMSE':np.sqrt(np.mean(g['Erro previsto−real']**2)),'Erro médio':g['Erro previsto−real'].mean()}),include_groups=False).reset_index()
display(resumo_erros); resumo_erros.to_csv(DIR_RESULTADOS/'erros_por_faixa.csv',index=False); residuos.to_csv(DIR_RESULTADOS/'residuos_teste_j7.csv',index=False,date_format='%Y-%m-%d')
fig,ax=plt.subplots(figsize=(8,4)); ax.axhline(0,color='black',lw=.8); ax.scatter(residuos['Real'],residuos['Erro previsto−real'],s=12,alpha=.4); ax.axvline(p10,ls='--',color='gray'); ax.axvline(p90,ls='--',color='gray'); ax.set(title='Resíduos no teste — J7, chuva até t',xlabel='Cota observada',ylabel='Previsto − observado'); ax.grid(alpha=.2); fig.tight_layout(); fig.savefig(DIR_FIGURAS/'residuos_niveis.png',dpi=160,bbox_inches='tight'); plt.show()

# Acumulados de chuva em janelas completas, sem preencher dias ausentes.
chuva_acc={w:diaria['Chuva'].rolling(w,min_periods=w).sum() for w in (7,15,30,60)}
chuva_tabela=pd.DataFrame({'Data':v['datas'],'Cota':v['real']})
for w,s in chuva_acc.items(): chuva_tabela[f'Chuva_{w}d']=[s.get(d-pd.Timedelta(days=1),np.nan) for d in v['datas']]
chuva_tabela.to_csv(DIR_RESULTADOS/'chuva_antecedente_teste.csv',index=False,date_format='%Y-%m-%d')
fig,axes=plt.subplots(2,2,figsize=(12,8));
for ax,w in zip(axes.flat,(7,15,30,60)):
    ax.scatter(chuva_tabela[f'Chuva_{w}d'],chuva_tabela['Cota'],s=10,alpha=.35); ax.set(title=f'Chuva acumulada nos {w} dias anteriores',xlabel='Precipitação acumulada (unidade da exportação ANA)',ylabel='Cota (unidade da exportação ANA)'); ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(DIR_FIGURAS/'chuva_acumulada_niveis.png',dpi=160,bbox_inches='tight'); plt.show()

# Episódios contínuos acima/abaixo dos percentis do treino; somente intervalos com cota observada.
serie_cota=diaria['Cota']; mask=(serie_cota.notna() & (serie_cota>=p90)) | (serie_cota.notna() & (serie_cota<=p10))
episodios=[]; grupo=None; tipo=None; anterior=None
for data,valor in serie_cota.items():
    novo='Alta' if pd.notna(valor) and valor>=p90 else ('Baixa' if pd.notna(valor) and valor<=p10 else None)
    if novo is None or (grupo is not None and (novo!=tipo or data-anterior!=pd.Timedelta(days=1))):
        if grupo: episodios.append({'Faixa':tipo,'Início':grupo[0][0],'Fim':grupo[-1][0],'Duração dias':len(grupo),'Cota mínima':min(x[1] for x in grupo),'Cota máxima':max(x[1] for x in grupo)})
        grupo=None; tipo=None
    if novo is not None:
        if grupo is None: grupo=[]; tipo=novo
        grupo.append((data,float(valor))); anterior=data
if grupo: episodios.append({'Faixa':tipo,'Início':grupo[0][0],'Fim':grupo[-1][0],'Duração dias':len(grupo),'Cota mínima':min(x[1] for x in grupo),'Cota máxima':max(x[1] for x in grupo)})
episodios=pd.DataFrame(episodios); episodios.to_csv(DIR_RESULTADOS/'episodios_p10_p90.csv',index=False,date_format='%Y-%m-%d')
display(episodios[episodios['Início'].dt.year.isin([2020,2022,2023,2024])] if len(episodios) else episodios)

print(f'P10 treino={p10:.3f}; P90 treino={p90:.3f}. São limites estatísticos, não operacionais.')
print('O erro positivo indica superestimação; erro negativo indica subestimação.')

### Interpretacao dos graficos

As janelas usam as mesmas datas de teste; unidades sao as numericas exportadas e nao confirmadas. A associacao entre acumulado de chuva e cota e descritiva, nao causal.

Fonte das figuras e tabelas: exportacoes locais ANA/HidroWeb; veja os CSVs e o manifesto em `resultados/v3/`. A comparacao e retrospectiva e a unidade da cota nao foi confirmada.

### Interpreta??o dos resultados da V4

Preencha esta interpreta??o somente depois de executar o notebook. Compare as m?tricas calculadas no teste, verifique o suporte de cada classe e descreva os erros em n?veis altos. A avalia??o ? explorat?ria porque o per?odo hist?rico j? foi analisado em vers?es anteriores.


## Epis?dios hist?ricos

Esta c?lula carrega a s?rie di?ria e as tabelas de epis?dios preparadas nas etapas anteriores para explorar per?odos de cotas altas e baixas. A tabela ? descritiva; n?o transforma associa??o de chuva e cota em causalidade.


In [ ]:
base = pd.read_csv(DIR_RESULTADOS / 'series_diarias.csv', index_col=0, parse_dates=True)
epis = pd.read_csv(DIR_RESULTADOS / 'episodios_p10_p90.csv')

# The original V2 labels include encoding damage; use column position for dates and group descriptors.
start_col, end_col = epis.columns[1], epis.columns[2]
epis[start_col] = pd.to_datetime(epis[start_col]); epis[end_col] = pd.to_datetime(epis[end_col])
rows = []
for _, event in epis.iterrows():
    start, end = event[start_col], event[end_col]
    interval = base.loc[start:end]
    peak_date = interval['Cota'].idxmax() if interval['Cota'].notna().any() else pd.NaT
    item = {'Faixa': event.iloc[0], 'Inicio': start, 'Fim': end, 'Duracao dias': int((end-start).days+1),
            'Data pico': peak_date, 'Cota maxima': interval['Cota'].max(), 'Vazao mediana': interval['Vazao'].median()}
    for window in (7, 15, 30):
        rain = base['Chuva'].rolling(window, min_periods=window).sum().shift(1)
        item[f'Chuva {window}d antes inicio'] = rain.get(start, np.nan)
        item[f'Chuva {window}d antes pico'] = rain.get(peak_date, np.nan) if pd.notna(peak_date) else np.nan
    rows.append(item)
event_rain = pd.DataFrame(rows)
event_rain.to_csv(DIR_RESULTADOS / 'episodios_com_chuva.csv', index=False, date_format='%Y-%m-%d')
summary=[]
for label,group in event_rain.groupby('Faixa'):
    summary.append({'Faixa':label,'N':len(group),'Median_duration_days':group.iloc[:,3].median(),'Rain7':group.iloc[:,7].median(),'Rain15':group.iloc[:,9].median(),'Rain30':group.iloc[:,11].median(),'N_rain30':group.iloc[:,11].count()})
display(pd.DataFrame(summary))
fig, ax = plt.subplots(figsize=(9, 5))
plot_values, plot_labels = [], []
for group in ('Baixa', 'Alta'):
    values = event_rain.loc[event_rain['Faixa'].eq(group), 'Chuva 30d antes inicio'].dropna().to_numpy()
    if len(values): plot_values.append(values); plot_labels.append(f'{group} (n={len(values)})')
if plot_values: ax.boxplot(plot_values, tick_labels=plot_labels)
ax.set(title='Chuva acumulada nos 30 dias antes dos episodios', ylabel='Unidade num?rica da exporta??o ANA')
ax.grid(axis='y', alpha=.2); fig.tight_layout(); fig.savefig(DIR_FIGURAS / 'chuva_30d_por_episodio.png', dpi=160); plt.show()

### Interpretacao desta etapa
Os acumulados de chuva descrevem condi??es anteriores aos epis?dios e n?o demonstram causalidade. Consulte o CSV para contagens e cobertura. Unidade e referencial da cota continuam sem confirma??o documental.
Fonte: CSVs di?rios ANA/HidroWeb usados nesta execu??o. A cobertura incompleta de chuva aparece como ausente; veja a coluna N_rain30 no CSV.

## Manifesto preliminar de arquivos

Esta etapa registra os artefatos presentes no diret?rio de resultados. O manifesto final ser? atualizado ap?s salvar tamb?m as m?tricas de classifica??o.


In [ ]:
# Manifesto de artefatos criados por esta execução.
arquivos_gerados=sorted(str(p.relative_to(RAIZ)) for p in DIR_RESULTADOS.rglob('*') if p.is_file())
arquivos_gerados.append(str((DIR_RESULTADOS/'manifesto_arquivos.txt').relative_to(RAIZ)))
(DIR_RESULTADOS/'manifesto_arquivos.txt').write_text('\n'.join(arquivos_gerados)+'\n',encoding='utf-8')
print('Figuras:'); print(*[p.name for p in sorted(DIR_FIGURAS.glob('*.png'))],sep='\n')
print('Reprodução: abrir este notebook no VS Code, selecionar o kernel do .venv e executar todas as células.')

## 9. Classificacao por regras

**O que faz:** Converte cotas observadas e previstas em faixas P10/P90 do treino. Precisao e acertos entre previsoes da classe; recall e casos reais encontrados; F1 resume ambos. E regra sobre regressao, nao outra rede. Unidade/referencial nao confirmados: nao aplicar cortes oficiais.

**Por que:** preparar ou interpretar a previsao diaria e os episodios do rio.

**Entradas/saidas e verificacao:** usa objetos/csvs mencionados na etapa; confira datas, contagens, lacunas, escalas e avisos antes de interpretar.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
v=pred_todas[7]
labels=['Baixa (<P10 treino)','Usual (P10-P90)','Alta (>P90 treino)']
def classificar(x): return np.select([x<p10,x>p90],[labels[0],labels[2]],default=labels[1])
reais,previstas=classificar(v['real']),classificar(v['pred'])
matriz=confusion_matrix(reais,previstas,labels=labels)
relatorio=pd.DataFrame(classification_report(reais,previstas,labels=labels,output_dict=True,zero_division=0)).T
display(pd.DataFrame(matriz,index=labels,columns=labels)); display(relatorio)
relatorio.to_csv(DIR_RESULTADOS/'metricas_classificacao_faixas.csv')
pd.DataFrame(matriz,index=labels,columns=labels).to_csv(DIR_RESULTADOS/'matriz_confusao_faixas.csv')
fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(matriz,cmap='Blues'); ax.set_xticks(range(3),labels,rotation=20,ha='right'); ax.set_yticks(range(3),labels)
for r in range(3):
 for c in range(3): ax.text(c,r,str(matriz[r,c]),ha='center',va='center')
ax.set(title='Faixas estat?sticas: observado ? previsto',xlabel='Prevista',ylabel='Observada'); fig.colorbar(im,ax=ax,label='Dias'); fig.tight_layout(); fig.savefig(DIR_FIGURAS/'matriz_confusao_faixas.png',dpi=160); plt.show()


### Interpreta??o dos resultados da V4

Preencha esta interpreta??o somente depois de executar o notebook. Compare as m?tricas calculadas no teste, verifique o suporte de cada classe e descreva os erros em n?veis altos. A avalia??o ? explorat?ria porque o per?odo hist?rico j? foi analisado em vers?es anteriores.


### Interpreta??o dos resultados da V4

Preencha esta interpreta??o somente depois de executar o notebook. Compare as m?tricas calculadas no teste, verifique o suporte de cada classe e descreva os erros em n?veis altos. A avalia??o ? explorat?ria porque o per?odo hist?rico j? foi analisado em vers?es anteriores.


## Faixas hidrol?gicas experimentais da V4

**O que faremos:** vamos classificar os valores reais e previstos do teste com os mesmos intervalos. Isso ? uma regra determin?stica aplicada ap?s a regress?o, n?o uma rede de classifica??o. A convers?o da unidade exportada para metros n?o foi validada nos metadados; portanto, os limites abaixo s? podem ser interpretados como hip?tese e n?o devem receber significado operacional.

**Par?metros:** P10 ? calculado apenas no treino; 5, 9, 12 e 15 s?o metros sob a hip?tese de 100 unidades CSV por metro. As fronteiras seguem `<5`, `[5,9)`, `[9,12)`, `[12,15]` e `>15`, com a classe baixa exigindo simultaneamente abaixo de P10 e abaixo de 5 m. Esperamos uma matriz e m?tricas por classe; classe sem suporte real n?o ? avali?vel.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, balanced_accuracy_score, f1_score

# Usa o protocolo de chuva at? t e a janela J7 da avalia??o comum.
v = previsoes[('t', 7, True)]
tr = v['treino']; te = v['teste']
# O P10 ? derivado apenas dos alvos do treino efetivo.
p10_csv = float(np.quantile(tr['y'], 0.10))
# Convers?o em metros ? uma hip?tese de trabalho, n?o confirmada para todo o hist?rico.
def classes_v4(x_csv):
    x_m = np.asarray(x_csv, dtype=float) / 100.0
    p10_m = p10_csv / 100.0
    return np.select([ (x_m < p10_m) & (x_m < 5), (x_m >= p10_m) & (x_m < 5), (x_m >= 5) & (x_m < 9), (x_m >= 9) & (x_m < 12), (x_m >= 12) & (x_m <= 15), x_m > 15 ], [0,1,2,3,4,5], default=-1)
rotulos=['Baixo experimental','Usual','Aten??o','Alerta','Acima de 12 at? 15 m','Extremo experimental']
y_real_cls=classes_v4(v['real']); y_pred_cls=classes_v4(v['pred'])
valid=(y_real_cls>=0)&(y_pred_cls>=0)
cm=confusion_matrix(y_real_cls[valid],y_pred_cls[valid],labels=np.arange(6))
print('P10 de treino (unidades CSV):',p10_csv)
print('Aviso: convers?o para metros n?o confirmada; classes s?o experimentais.')
print('Acur?cia:',accuracy_score(y_real_cls[valid],y_pred_cls[valid]))
print('Balanced accuracy:',balanced_accuracy_score(y_real_cls[valid],y_pred_cls[valid]))
print('Macro-F1:',f1_score(y_real_cls[valid],y_pred_cls[valid],average='macro',zero_division=0))
display(pd.DataFrame(cm,index=rotulos,columns=rotulos))
rel=classification_report(y_real_cls[valid],y_pred_cls[valid],labels=np.arange(6),target_names=rotulos,output_dict=True,zero_division=0)
display(pd.DataFrame(rel).T)


## Exportar tabela de classes e manifesto

**O que faz:** salva a matriz de confus?o, o relat?rio por classe e a lista dos arquivos gerados em `resultados/v4/`. O manifesto facilita conferir a execu??o. Esperamos arquivos CSV e PNG criados pelas etapas anteriores; a lista s? reflete o que realmente existe ao final da execu??o.


In [ ]:
pd.DataFrame(cm,index=rotulos,columns=rotulos).to_csv(DIR_RESULTADOS/'matriz_confusao_v4.csv',encoding='utf-8')
pd.DataFrame(rel).T.to_csv(DIR_RESULTADOS/'metricas_classificacao_v4.csv',encoding='utf-8')
arquivos=sorted(str(x.relative_to(RAIZ)) for x in DIR_RESULTADOS.rglob('*') if x.is_file() and x.name!='manifesto.txt')
(DIR_RESULTADOS/'manifesto.txt').write_text('\n'.join(arquivos)+'\n',encoding='utf-8')
print('Manifesto salvo:',DIR_RESULTADOS/'manifesto.txt')
